In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore') # yfinance prints noisy warnings we can ignore

# 1. Define the asset
ticker_symbol = "SPY"
spy = yf.Ticker(ticker_symbol)

# 2. Get the current risk-free rate (13-week Treasury Bill proxy)
irx = yf.Ticker("^IRX")
rfr_history = irx.history(period="1d")
# ^IRX is quoted in percent (e.g., 5.25), so we divide by 100
risk_free_rate = rfr_history['Close'].iloc[-1] / 100 
print(f"Current Risk-Free Rate: {risk_free_rate:.4%}")

# 3. Fetch all expiration dates
expirations = spy.options
print(f"Found {len(expirations)} expiration dates.")

# 4. Loop through expirations and collect Call options
calls_list = []
today = datetime.today()

for exp in expirations:
    try:
        # Get the option chain for this expiration
        chain = spy.option_chain(exp)
        calls = chain.calls
        
        # Calculate Time to Maturity (Tau) in years
        exp_date = datetime.strptime(exp, '%Y-%m-%d')
        days_to_expiry = (exp_date - today).days
        
        # Skip options expiring today to avoid division-by-zero later
        if days_to_expiry <= 0:
            continue
            
        tau = days_to_expiry / 365.25
        calls['tau'] = tau
        calls['expiration'] = exp
        
        calls_list.append(calls)
    except Exception as e:
        print(f"Skipped {exp} due to missing data.")

# 5. Combine into a single DataFrame
df_calls = pd.concat(calls_list, ignore_index=True)

# 6. Calculate the Mid-Price
# Quants rarely use 'lastPrice' because options trade infrequently. 
# The fair market price is the average of the bid and ask.
df_calls['mid_price'] = (df_calls['bid'] + df_calls['ask']) / 2

# 7. Get the current underlying stock price (S0)
S0 = spy.history(period="1d")['Close'].iloc[-1]
df_calls['S0'] = S0
df_calls['r'] = risk_free_rate

# 8. Filter for options with actual liquidity (volume > 0 and bid > 0)
df_calls = df_calls[(df_calls['volume'] > 0) & (df_calls['bid'] > 0)]

# Save to the data folder
output_path = "../data/spy_call_options.csv"
df_calls.to_csv(output_path, index=False)
print(f"Successfully saved {len(df_calls)} option contracts to {output_path}")

Current Risk-Free Rate: 3.7070%
Found 32 expiration dates.
Successfully saved 4372 option contracts to ../data/spy_call_options.csv
